In [172]:
# Loading packages
import numpy as np
import pandas as pd
import arviz as az
import seaborn as sns
import matplotlib.pyplot as plt
import pymc as pm
from pathlib import Path
from scipy import stats
import statsmodels.api as sm
import scipy as sc

# Loading data
data_path = Path.cwd()
while not (data_path / "data" / "fg_builds").exists() and data_path != data_path.parent:
    data_path = data_path.parent

train_path = data_path / "data" / "fg_builds" / "fg_hitters_train.csv"
test_path = data_path / "data" / "fg_builds" / "fg_hitters_test.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

In [173]:
train.head()

,IDfg,Season,Name,Team,Age,G,AB,PA,H,1B,...,OAA_P,FRM_y,primary_pos,GS,GS_pct,same_team,rookie,team_start,team_end,role
0,10155,2016,Mike Trout,LAA,24,159,549,681,173,107,...,NaN,NaN,CF,157.0,0.987421,0.0,NaN,LAA,LAA,starter
1,15429,2016,Kris Bryant,CHC,24,155,603,699,176,99,...,NaN,NaN,3B,155.0,1.000000,0.0,NaN,CHC,CHC,starter
2,13611,2016,Mookie Betts,BOS,23,158,672,730,214,136,...,NaN,NaN,RF,157.0,0.993671,0.0,NaN,BOS,BOS,starter
3,5417,2016,Jose Altuve,HOU,26,161,640,717,216,145,...,NaN,NaN,2B,160.0,0.993789,0.0,NaN,HOU,HOU,starter
4,5038,2016,Josh Donaldson,TOR,30,155,577,700,164,90,...,NaN,NaN,3B,154.0,0.993548,0.0,NaN,TOR,TOR,starter


In [174]:
BASE_COLS = ["Name", "G", "PA", "IDfg", "Age", "Team", "Season", "wRC+"]
PRIOR_SEASON_PRED = ["PA", "BB%", "K%", "Pull%", "Hard%",
                      "Barrel%", "GB%", "FB%", "maxEV", "wRC+", "primary_pos"]

# Continuous vars that get a t1 value AND a (t1 - t2) delta.
# PA is excluded (we keep raw t2 PA instead of a delta).
# primary_pos is excluded (categorical — no meaningful delta).
DELTA_VARS = [v for v in PRIOR_SEASON_PRED if v not in ("PA", "primary_pos")]


def build_regression_features(df, prior_source):
    """
    For each row in df, builds:
      - BASE_COLS from the target season
      - {col}_t1  : PRIOR_SEASON_PRED value from Season - 1
      - {col}_delta: (Season-1) minus (Season-2) for continuous PRIOR_SEASON_PRED vars
      - PA_t2     : raw PA from Season - 2 (not a delta, per spec)
      - primary_pos_t1 is included but has no delta (categorical)

    Missing prior seasons produce NaN — fine per spec.

    df           : target dataset to build features for (train or test)
    prior_source : dataset to look up prior seasons from
                   (train for both train and test features)
    """
    base = df[BASE_COLS].copy()

    # Deduplicated lookup indexed by (IDfg, Season)
    prior_cols = ["IDfg", "Season"] + PRIOR_SEASON_PRED
    prior_df = prior_source[prior_cols].drop_duplicates(subset=["IDfg", "Season"])

    # t-1: advance Season by 1 so a 2022 row joins onto target Season 2023
    t1 = prior_df.copy()
    t1["Season"] = t1["Season"] + 1
    t1 = t1.rename(columns={col: f"{col}_t1" for col in PRIOR_SEASON_PRED})

    # t-2: advance Season by 2
    t2 = prior_df.copy()
    t2["Season"] = t2["Season"] + 2
    t2 = t2.rename(columns={col: f"{col}_t2" for col in PRIOR_SEASON_PRED})

    result = (
        base
        .merge(t1, on=["IDfg", "Season"], how="left")
        .merge(
            t2[["IDfg", "Season"] + [f"{col}_t2" for col in PRIOR_SEASON_PRED]],
            on=["IDfg", "Season"],
            how="left",
        )
    )

    # Compute (t1 - t2) deltas for continuous vars
    for col in DELTA_VARS:
        result[f"{col}_delta"] = result[f"{col}_t1"] - result[f"{col}_t2"]

    # Drop intermediate t2 columns; keep PA_t2 (raw, per spec)
    t2_drop = [f"{col}_t2" for col in PRIOR_SEASON_PRED if col != "PA"]
    result = result.drop(columns=t2_drop)

    # Additional helpful features
    result["Age2"] = result["Age"]**2
    result["x0"] = np.ones(result.shape[0])
    result["BB-K%_t1"] = result["BB%_t1"] - result["K%_t1"]
    result["Barrel%_t1_log"] = np.log(result["Barrel%_t1"]*100 + 0.01)

    return result


# Train features: prior seasons sourced from train itself.
# Filter to Season >= 2017 AFTER the merge so 2016 rows can still serve as t-1.
train_reg = build_regression_features(train, train)
train_reg = train_reg[train_reg["Season"] >= 2017].reset_index(drop=True)

# Test features: prior seasons come from train (test is 2024, so t-1=2023, t-2=2022).
test_reg = build_regression_features(test, train)

print(f"train_reg: {train_reg.shape}  |  seasons: {sorted(train_reg['Season'].unique())}")
print(f"test_reg:  {test_reg.shape}   |  seasons: {sorted(test_reg['Season'].unique())}")
train_reg.head()

train_reg: (4476, 33)  |  seasons: [2017, 2018, 2019, 2020, 2021, 2022, 2023]
test_reg:  (650, 33)   |  seasons: [2024]


,Name,G,PA,IDfg,Age,Team,Season,wRC+,PA_t1,BB%_t1,...,Hard%_delta,Barrel%_delta,GB%_delta,FB%_delta,maxEV_delta,wRC+_delta,Age2,x0,BB-K%_t1,Barrel%_t1_log
0,Aaron Judge,155,678,15640,25,NYY,2017,174.0,95.0,0.095,...,NaN,NaN,NaN,NaN,NaN,NaN,625,1.0,-0.347,2.451867
1,Jose Altuve,153,662,5417,27,HOU,2017,160.0,717.0,0.084,...,NaN,NaN,NaN,NaN,NaN,NaN,729,1.0,-0.014,1.932970
2,Jose Ramirez,152,645,13510,24,CLE,2017,146.0,618.0,0.071,...,NaN,NaN,NaN,NaN,NaN,NaN,576,1.0,-0.029,1.101940
3,Kris Bryant,151,665,15429,25,CHC,2017,147.0,699.0,0.107,...,NaN,NaN,NaN,NaN,NaN,NaN,625,1.0,-0.113,2.460443
4,Mike Trout,114,507,10155,25,LAA,2017,180.0,681.0,0.170,...,NaN,NaN,NaN,NaN,NaN,NaN,625,1.0,-0.031,2.646884


# Bayesian Linear Regression

In [175]:
def BLR(X,y,mu_0,Om_0_inv,a_0,b_0,ind_names,N):
    col_names = ['posterior mean','lower 95% bound','upper 95% bound']
    n,p = X.shape
    XtX = X.T.dot(X)
    Om_n_inv = XtX + Om_0_inv
    Om_n = sc.linalg.inv(Om_n_inv)
    term1 = Om_0_inv.dot(mu_0)+X.T.dot(y)
    mu_n = Om_n.dot(term1)
    a_n = a_0 + n/2
    term2 = y.T.dot(y)+mu_0.dot(Om_0_inv.dot(mu_0))+mu_n.dot(Om_n_inv.dot(mu_n))
    b_n = b_0 + term2/2
    sigma2 = 1/np.random.gamma(a_n, 1/b_n, N)
    betas = np.zeros((N,p))
    # draw N samples from the marginal posterior of beta
    for i in range(0,N):
        s2 =sigma2[i]
        cov = s2*Om_n
        betas[i,] = np.random.multivariate_normal(mu_n,cov,1)

    #find the mean of each column which corresponds to each beta coefficient     
    mu_beta = np.mean(betas, axis=0)
    #find the 2.5 and 97.5 percentils which correspond to each beta coefficient 
    lower95 = np.percentile(betas,2.5,axis=0)
    upper95 = np.percentile(betas,97.5,axis=0)
    results = np.column_stack([mu_beta,lower95,upper95])
    results = pd.DataFrame(results,columns = col_names,index=ind_names)
    return results, mu_n, Om_n, a_n, b_n

def ModEvidence(X,y,mu_n,Om_n,mu_0,Om_0_inv,a_n,b_n,a_0,b_0): 
    #set beta and sigma2 to their posterior mean 
    beta = mu_n
    sigma2 = b_n/(a_n-1)
    Om_0 = sc.linalg.inv(Om_0_inv)
    mu = X.dot(beta)
    cov = sigma2*np.eye(X.shape[0])
    ll = sc.stats.multivariate_normal.logpdf(y,mu,cov)
    # evaluate log-prior
    lprior = a_0*np.log(b_0)-sc.special.loggamma(a_0)- (a_0+1)*np.log(sigma2)-b_0/sigma2
    lprior = lprior + sc.stats.multivariate_normal.logpdf(beta,mu_0,sigma2*Om_0)
    # evaluate log-posterior
    lpost = a_n*np.log(b_n)-sc.special.loggamma(a_n)- (a_n+1)*np.log(sigma2)-b_n/sigma2
    lpost = lpost +sc.stats.multivariate_normal.logpdf(beta,mu_n,sigma2*Om_n)
    lmodevid = ll+lprior-lpost
    return lmodevid

## Model 1: "Naive" Approach

- No shrinkage of noisy predictors
- No adjustment for heteroskedasticity
- 100 PA minimum in target and predicted years

In [176]:
def naive_BLR_spec(predictors, target_pa=100, previous_pa=100, old_pa = 0):
    # Set PA thresholds (100 PA default)
    df = train_reg[(train_reg["PA"] >= target_pa) & (train_reg["PA_t1"] >= previous_pa) & (train_reg["PA_t2"] >= old_pa)]

    X = df[predictors]
    y = df['wRC+']
    n,p = X.shape

    #set up prior parameters
    mu_0 = np.zeros(p)
    XtX = X.T.dot(X)
    Om_0_inv = XtX/n #unit information prior
    a_0 = 0.01
    b_0 = 0.01

    N=10000 #Monte Carlo sample size

    results, mu_n, Om_n, a_n, b_n = BLR(X,y,mu_0,Om_0_inv,a_0,b_0,predictors,N)
    log_evid= ModEvidence(X,y,mu_n,Om_n,mu_0,Om_0_inv,a_n,b_n,a_0,b_0)

    return(results, log_evid)

In [177]:
## Set of candidate models

# Simple previous year wRC+ model
model1, evid1 = naive_BLR_spec(["x0", "wRC+_t1"])

# Simple "core" metric models
model2, evid2 = naive_BLR_spec(["x0", "K%_t1", "BB%_t1", "Barrel%_t1"])
model3, evid3 = naive_BLR_spec(["x0", "BB-K%_t1", "Barrel%_t1"])
model4, evid4 = naive_BLR_spec(["x0", "Age", "Age2", "K%_t1", "BB%_t1", "Barrel%_t1"])
model5, evid5 = naive_BLR_spec(["x0", "Age", "Age2", "BB-K%_t1", "Barrel%_t1"])

# More complex metric models
model6, evid6 = naive_BLR_spec(["x0", "Age", "Age2", "K%_t1", "BB%_t1", "Hard%_t1", "Pull%_t1", "FB%_t1"])
model7, evid7 = naive_BLR_spec(["x0", "Age", "Age2", "BB-K%_t1", "Hard%_t1", "Pull%_t1", "FB%_t1"])

# More complex wRC+ models
model8, evid8 = naive_BLR_spec(["x0", "wRC+_t1", "wRC+_delta"])
model9, evid9 = naive_BLR_spec(["x0", "Age", "Age2", "wRC+_t1"])
model10, evid10 = naive_BLR_spec(["x0", "Age", "Age2", "wRC+_t1", "wRC+_delta"], old_pa=1)


# Higher previous year PA thresholds 

# Simple previous year wRC+ model
model1b, evid1b = naive_BLR_spec(["x0", "wRC+_t1"], previous_pa=250)

# Simple "core" metric models
model2b, evid2b = naive_BLR_spec(["x0", "K%_t1", "BB%_t1", "Barrel%_t1"], previous_pa=250)
model3b, evid3b = naive_BLR_spec(["x0", "BB-K%_t1", "Barrel%_t1"], previous_pa=250)
model4b, evid4b = naive_BLR_spec(["x0", "Age", "Age2", "K%_t1", "BB%_t1", "Barrel%_t1"], previous_pa=250)
model5b, evid5b = naive_BLR_spec(["x0", "Age", "Age2", "BB-K%_t1", "Barrel%_t1"], previous_pa=250)

# More complex metric models
model6b, evid6b = naive_BLR_spec(["x0", "Age", "Age2", "K%_t1", "BB%_t1", "Hard%_t1", "Pull%_t1", "FB%_t1"], previous_pa=250)
model7b, evid7b = naive_BLR_spec(["x0", "Age", "Age2", "BB-K%_t1", "Hard%_t1", "Pull%_t1", "FB%_t1"], previous_pa=250)

# More complex wRC+ models
model8b, evid8b = naive_BLR_spec(["x0", "wRC+_t1", "wRC+_delta"], previous_pa=250)
model9b, evid9b = naive_BLR_spec(["x0", "Age", "Age2", "wRC+_t1"], previous_pa=250)
model10b, evid10b = naive_BLR_spec(["x0", "Age", "Age2", "wRC+_t1", "wRC+_delta"], previous_pa=250, old_pa=1)

In [178]:
evidence = pd.DataFrame({"model" : [1,2,3,4,5,6,7,8,9, 10],
                        "evidence" : [evid1, evid2, evid3, evid4, evid5,
                                       evid6, evid7, evid8, evid9, evid10]})
evidence

,model,evidence
0,1,-10335.040153
1,2,-10342.463253
2,3,-10338.730585
3,4,-10349.904141
4,5,-10346.171314
5,6,-10357.431399
6,7,-10353.702480
7,8,-10338.745808
8,9,-10342.487995
9,10,-10346.190308


In [179]:
evidenceb = pd.DataFrame({"model" : [1,2,3,4,5,6,7,8,9, 10],
                        "evidence" : [evid1b, evid2b, evid3b, evid4b, evid5b,
                                       evid6b, evid7b, evid8b, evid9b, evid10b]})
evidenceb

,model,evidence
0,1,-7442.341927
1,2,-7449.474256
2,3,-7445.907574
3,4,-7456.593199
4,5,-7453.026442
5,6,-7463.774677
6,7,-7460.209673
7,8,-7445.900944
8,9,-7449.469006
9,10,-7453.026144


## Model Evaluation: 2024 predictions, RMSE

In [180]:
# Deterministic MARCEL 

marcel_det = train[train["Season"] >= 2021][["IDfg", "Name", "Season", "Age", "primary_pos", "wRC+", "PA"]]
marcel_det["weight"] = (marcel_det["Season"] - 2021 + 3)*marcel_det["PA"] / 1000
marcel_det["weighted_wrc"] = marcel_det["weight"]*marcel_det["wRC+"]

marcel_det_preds = pd.DataFrame(marcel_det.groupby(["IDfg"], group_keys=False).apply(lambda g: (((g.weighted_wrc).sum() + 120)/((g.weight).sum() + 1.2))*(1 - 0.006*((g.Age.max() + 1) - 29)) if g.Age.max() + 1 >= 29
                                        else (((g.weighted_wrc).sum() + 120)/((g.weight).sum() + 1.2))*(1 + 0.003*(29 - (g.Age.max() + 1)))).reset_index())
marcel_det_preds.columns = ["IDfg", "MARCEL_wRC+"]

# Sum of raw weights across all seasons for each player (before regression-to-mean adjustment)
weight_sums = marcel_det.groupby("IDfg")["weight"].sum()
marcel_det_preds["weight_sum"] = marcel_det_preds["IDfg"].map(weight_sums)          

eval_df = test.join(marcel_det_preds.set_index("IDfg"),
                       on = "IDfg")[["IDfg", "Name", 
                                     "Age", "primary_pos",
                                     "G", "PA",
                                     "wRC+", "MARCEL_wRC+"]]
eval_df["MARCEL_wRC+"] = eval_df["MARCEL_wRC+"]

/var/folders/3f/5z6bhnqn1fv15360tr2wfskr0000gn/T/ipykernel_39206/3028120742.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  marcel_det_preds = pd.DataFrame(marcel_det.groupby(["IDfg"], group_keys=False).apply(lambda g: (((g.weighted_wrc).sum() + 120)/((g.weight).sum() + 1.2))*(1 - 0.006*((g.Age.max() + 1) - 29)) if g.Age.max() + 1 >= 29


In [181]:
# Bayesian MARCEL
import sys
sys.path.insert(0, '../../../src')

import numpy as np
import pandas as pd
import pickle
import arviz as az

def load_and_filter(path: str, min_pa: int = 100) -> pd.DataFrame:
    """Load a pre-cleaned FanGraphs hitters CSV and apply the PA filter.

    Parameters
    ----------
    path   : path to fg_hitters_train.csv (or similar pre-cleaned file).
    min_pa : minimum plate appearances required to include a player-season.
             Default 100. Applied to all model-fitting rows (Block 1 and
             Block 2 prior seasons).

    Returns a long-format DataFrame with one row per qualifying (player, season).
    """
    df = pd.read_csv(path)
    df = df[df["PA"] >= min_pa]

    cols = ["IDfg", "Season", "Name", "Age", "PA", "wRC+", "primary_pos"]
    df = df[cols].reset_index(drop=True)

    df["player_season"] = df["IDfg"].astype(str) + "_" + df["Season"].astype(str)

    return df

# ── 1. Load trace ────────────────────────────────────────────────────────────
with open('block3/trace_2s_quad_100PA.pkl', 'rb') as f:
    trace = pickle.load(f)

# ── 2. Build index map from the trace's own player_season coordinates ────────
ps_coords = trace.posterior.coords["player_season"].values
n_ps      = len(ps_coords)
ps_to_idx = {ps: i for i, ps in enumerate(ps_coords)}

coord_df = pd.DataFrame({
    "IDfg":   [int(ps.rsplit("_", 1)[0]) for ps in ps_coords],
    "Season": [int(ps.rsplit("_", 1)[1]) for ps in ps_coords],
    "ps_idx": np.arange(n_ps),
})

df   = train[(train["Season"] <= 2022) | (train["PA"] >= 100)]
meta = df[["IDfg", "Season", "Name", "Age", "PA"]].copy()  # ADDED PA to metadata
coord_df = coord_df.merge(meta, on=["IDfg", "Season"], how="left")

# ── 3. Build 2024 projection roster ─────────────────────────────────────────
TARGET = 2024

players_2023 = (
    coord_df[coord_df["Season"] == TARGET - 1]
    [["IDfg", "ps_idx", "Name", "Age", "PA"]]  # ADDED PA
    .copy()
    .rename(columns={"ps_idx": "idx_t1", "PA": "PA_t1"})  # rename PA
)
players_2023["Age_2024"] = players_2023["Age"].astype(int) + 1

idx_t2_map = (
    coord_df[coord_df["Season"] == TARGET - 2]
    .set_index("IDfg")["ps_idx"]
)
pa_t2_map = (
    coord_df[coord_df["Season"] == TARGET - 2]
    .set_index("IDfg")["PA"]
)

players_2023["idx_t2"] = players_2023["IDfg"].map(idx_t2_map).fillna(-1).astype(int)
players_2023["PA_t2"]  = players_2023["IDfg"].map(pa_t2_map)  # NEW

players_2023["n_prior"] = (
    (players_2023["idx_t1"] != -1).astype(int)
    + (players_2023["idx_t2"] != -1).astype(int)
)

proj_df = players_2023[players_2023["n_prior"] > 0].reset_index(drop=True)

# ── 3b. Compute prior-PA proxy for predictive noise scale ────────────────────
# Use the most recent available prior season's PA.
# This is leakage-free: only uses training data (2022 or 2023 PA).
proj_df["PA_prior"] = proj_df["PA_t1"].fillna(proj_df["PA_t2"])
# Sanity check: every player should have PA_prior since n_prior > 0
assert proj_df["PA_prior"].notna().all(), "Some players missing prior PA"

print(
    f"Projecting {len(proj_df)} players for 2024 — "
    f"{(proj_df['n_prior']==2).sum()} two-season, "
    f"{(proj_df['n_prior']==1).sum()} one-season"
)
print(f"Prior-PA distribution (used for noise scale):")
print(proj_df["PA_prior"].describe().round(0))

# ── 4. Extract posterior samples ─────────────────────────────────────────────
theta_samp     = trace.posterior["theta"].values.reshape(-1, n_ps)
w_samp         = trace.posterior["w"].values.flatten()
a_0_samp       = trace.posterior["a_0"].values.flatten()
by_samp        = trace.posterior["beta_young"].values.flatten()
bo_samp        = trace.posterior["beta_old"].values.flatten()
sigma_obs_samp = trace.posterior["sigma_obs"].values.flatten()  # NEW

n_samp = w_samp.shape[0]

# ── 5. Vectorised projection (unchanged) ─────────────────────────────────────
idx_t1 = proj_df["idx_t1"].values.astype(int)
idx_t2 = proj_df["idx_t2"].values.astype(int)
ages   = proj_df["Age_2024"].values.astype(float)

safe_t1 = np.where(idx_t1 == -1, 0, idx_t1)
safe_t2 = np.where(idx_t2 == -1, 0, idx_t2)
mask1   = (idx_t1 != -1).astype(float)
mask2   = (idx_t2 != -1).astype(float)

theta_t1 = theta_samp[:, safe_t1]
theta_t2 = theta_samp[:, safe_t2]

w   = w_samp[:, None]
rw1 = w * mask1[None, :]
rw2 = (1.0 - w) * mask2[None, :]
theta_proj = (rw1 / (rw1 + rw2)) * theta_t1 + (rw2 / (rw1 + rw2)) * theta_t2

age_diff   = ages[None, :] - a_0_samp[:, None]
age_effect = np.where(
    age_diff >= 0,
    -bo_samp[:, None] * age_diff**2,
    -by_samp[:, None] * age_diff**2,
)

theta_proj_aged = theta_proj + age_effect  # shape: (n_samp, n_players)

# ── 5b. Compute predictive distribution using prior-PA proxy ─────────────────
# Add observation noise: ε ~ Normal(0, sigma_obs / sqrt(PA_prior))
# This is the distribution of observed wRC+ given the player's projected
# true talent, assuming 2024 playing time is similar to prior years.

pa_prior_arr = proj_df["PA_prior"].values.astype(float)  # shape: (n_players,)

# obs_sd: per-draw, per-player noise SD
# sigma_obs_samp shape: (n_samp,) -> (n_samp, 1)
# pa_prior_arr shape: (n_players,) -> (1, n_players)
obs_sd = sigma_obs_samp[:, None] / np.sqrt(pa_prior_arr[None, :])
# obs_sd shape: (n_samp, n_players)

rng = np.random.default_rng(42)
noise = rng.normal(0, obs_sd)  # same shape as theta_proj_aged

wrc_predictive = theta_proj_aged + noise

# ── 6. Posterior summaries ───────────────────────────────────────────────────
# True-talent HDI (your original calculation)
hdi_theta = az.hdi(theta_proj_aged, hdi_prob=0.94)

# Predictive HDI (true talent + observation noise)
hdi_predictive = az.hdi(wrc_predictive, hdi_prob=0.94)

projections = proj_df[["IDfg", "Name", "Age_2024", "n_prior", "PA_prior"]].copy()
projections["proj_wrc_mean"]   = theta_proj_aged.mean(axis=0).round(1)
projections["proj_wrc_sd"]     = theta_proj_aged.std(axis=0).round(1)

# True-talent HDI
projections["theta_hdi_lo"] = hdi_theta[:, 0].round(1)
projections["theta_hdi_hi"] = hdi_theta[:, 1].round(1)

# Predictive HDI (use this for coverage evaluation)
projections["pred_hdi_lo"] = hdi_predictive[:, 0].round(1)
projections["pred_hdi_hi"] = hdi_predictive[:, 1].round(1)
projections["pred_sd"]     = wrc_predictive.std(axis=0).round(1)

bmc_projections = projections.sort_values(
    "proj_wrc_mean", ascending=False
).reset_index(drop=True)

print("\nTop 20 projected wRC+ for 2024:")
print(bmc_projections.head(20).to_string(index=False))

/var/folders/3f/5z6bhnqn1fv15360tr2wfskr0000gn/T/ipykernel_39206/429193308.py:153: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi_theta = az.hdi(theta_proj_aged, hdi_prob=0.94)


Projecting 461 players for 2024 — 353 two-season, 108 one-season
Prior-PA distribution (used for noise scale):
count    461.0
mean     381.0
std      180.0
min      100.0
25%      226.0
50%      368.0
75%      528.0
max      753.0
Name: PA_prior, dtype: float64

Top 20 projected wRC+ for 2024:
 IDfg             Name  Age_2024  n_prior  PA_prior  proj_wrc_mean  proj_wrc_sd  theta_hdi_lo  theta_hdi_hi  pred_hdi_lo  pred_hdi_hi  pred_sd
15640      Aaron Judge        32        2       458          149.5          9.0         132.7         166.4        112.0        186.3     19.9
19556   Yordan Alvarez        27        2       496          145.1          9.0         128.0         161.3        111.2        184.1     19.4
19755    Shohei Ohtani        29        2       599          142.9          8.4         127.9         159.0        108.2        175.7     17.8
13611     Mookie Betts        31        2       693          137.9          7.8         123.1         152.8        108.4        168.5

/var/folders/3f/5z6bhnqn1fv15360tr2wfskr0000gn/T/ipykernel_39206/429193308.py:156: FutureWarning: hdi currently interprets 2d data as (draw, shape) but this will change in a future release to (chain, draw) for coherence with other functions
  hdi_predictive = az.hdi(wrc_predictive, hdi_prob=0.94)


In [182]:
bmc_projections["BMC_wRC+"] = bmc_projections["proj_wrc_mean"]
eval_df = eval_df.merge(bmc_projections[["IDfg", "BMC_wRC+"]], left_on = "IDfg", right_on = "IDfg")

In [183]:
## Predicting 2024 values using best models in BLR Ensemble: Models 1, 2, 3, 8

# ── Predict 2024 wRC+ using posterior means from each BLR model ──────────────
models = {
    "BLR1":  model1,  "BLR2":  model2,  "BLR3":  model3, "BLR8": model8,
    "BLR5":  model5, "BLR5b":  model5b, "BLR6":  model6, "BLR6b":  model6b, 
    "BLR1b":  model1b,  "BLR2b":  model2b,  "BLR3b":  model3b, "BLR8b": model8b
}

pred_df = test_reg[["IDfg"]].copy()

for label, model_df in models.items():
    predictors = model_df.index.tolist()          # e.g. ["x0", "wRC+_t1"]
    beta       = model_df["posterior mean"].values

    X = test_reg[predictors]

    # Rows where any predictor is NaN get NaN — don't silently use partial data
    valid  = X.notna().all(axis=1)
    y_pred = np.full(len(test_reg), np.nan)
    y_pred[valid] = X[valid].values @ beta

    pred_df[label] = np.round(y_pred, 1)

print(pred_df.head(10).to_string(index=False))

 IDfg  BLR1  BLR2  BLR3  BLR8  BLR5  BLR5b  BLR6  BLR6b  BLR1b  BLR2b  BLR3b  BLR8b
15640 130.5 163.6 164.8 140.3 160.6  163.5 135.5  136.5  137.9  166.4  167.1  145.6
25764 104.7 113.9 112.5 103.6 121.2  121.2 114.5  114.0  105.7  115.8  112.9  104.7
19755 133.7 139.8 140.5 134.8 140.1  142.6 120.1  121.7  141.8  142.0  142.2  142.6
20123 121.9 132.4 134.1 124.9 141.9  145.3 134.9  136.7  127.2  135.3  137.4  129.5
26289 107.4 105.1 105.2 109.7 115.6  113.9 114.6  112.6  109.1  104.4  104.3  110.9
12916 106.5 111.2 111.0 108.5 109.2  110.5 103.5  104.0  108.0  112.2  111.6  109.5
24617 107.4  84.1  84.1 103.3  86.1   84.9  96.0   96.2  109.1   82.1   82.4  105.7
26668  90.7  83.0  83.7   NaN  95.7   91.0  94.2   89.9   88.2   78.8   80.5    NaN
13510 107.4 115.4 115.2 111.4 113.3  116.8 106.8  108.4  109.1  119.0  117.9  112.2
13613 109.2 109.7 109.9 108.1 108.6  110.8 110.6  112.0  111.3  111.4  111.3  110.4


In [184]:
eval_df = eval_df.merge(pred_df, left_on = "IDfg", right_on = "IDfg")

In [185]:
eval_df

,IDfg,Name,Age,primary_pos,G,PA,wRC+,MARCEL_wRC+,BMC_wRC+,BLR1,...,BLR3,BLR8,BLR5,BLR5b,BLR6,BLR6b,BLR1b,BLR2b,BLR3b,BLR8b
0,15640,Aaron Judge,32,CF,158,704,220.0,165.151422,149.5,130.5,...,164.8,140.3,160.6,163.5,135.5,136.5,137.9,166.4,167.1,145.6
1,25764,Bobby Witt Jr.,24,SS,161,709,169.0,108.615999,108.0,104.7,...,112.5,103.6,121.2,121.2,114.5,114.0,105.7,115.8,112.9,104.7
2,19755,Shohei Ohtani,29,NaN,159,731,180.0,150.972881,142.9,133.7,...,140.5,134.8,140.1,142.6,120.1,121.7,141.8,142.0,142.2,142.6
3,20123,Juan Soto,25,RF,157,713,181.0,148.664292,134.2,121.9,...,134.1,124.9,141.9,145.3,134.9,136.7,127.2,135.3,137.4,129.5
4,26289,Gunnar Henderson,23,SS,159,719,154.0,119.196480,113.7,107.4,...,105.2,109.7,115.6,113.9,114.6,112.6,109.1,104.4,104.3,110.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
407,6887,Martin Maldonado,37,C,48,147,11.0,69.462367,74.6,82.1,...,81.9,80.2,72.8,67.7,75.0,71.1,77.5,76.9,78.4,75.9
408,15676,Jose Abreu,37,1B,35,120,0.0,106.836785,93.7,91.1,...,99.5,96.6,91.7,89.5,80.3,79.4,88.8,99.7,98.8,93.2
409,20543,Elehuris Montero,25,1B,67,247,47.0,86.823776,89.3,87.0,...,70.3,85.8,74.7,69.9,93.9,90.6,83.7,64.4,65.6,82.7
410,12155,Eddie Rosario,32,LF,91,319,44.0,92.178484,93.0,97.9,...,98.8,93.1,93.9,93.3,88.5,88.0,97.2,98.7,97.7,93.3


In [186]:
def model_eval(min_PA=0):
    df = eval_df[eval_df["PA"] > min_PA]

    exclude = {"IDfg", "Name", "Age", "primary_pos", "G", "PA", "wRC+"}
    pred_cols = [c for c in df.columns if c not in exclude]

    actual = df["wRC+"]
    pa     = df["PA"]
    pa_sum = pa.sum()

    rows = []
    for col in pred_cols:
        residuals = actual - df[col]
        rmse      = np.sqrt((residuals ** 2).mean())
        mae       = residuals.abs().mean()
        rmse_pa   = np.sqrt((pa * residuals ** 2).sum() / pa_sum)
        mae_pa    = (pa * residuals.abs()).sum() / pa_sum
        rows.append({
            "model":    col,
            "RMSE":     round(rmse,    2),
            "MAE":      round(mae,     2),
            "RMSE_PA":  round(rmse_pa, 2),
            "MAE_PA":   round(mae_pa,  2),
            "n":        len(df),
        })

    rmse_table = (
        pd.DataFrame(rows)
        .sort_values("RMSE")
        .reset_index(drop=True)
    )
    print(rmse_table.to_string(index=False))

In [187]:
model_eval(min_PA=0)

      model  RMSE   MAE  RMSE_PA  MAE_PA   n
   BMC_wRC+ 39.48 25.45    24.30   18.38 411
MARCEL_wRC+ 39.55 25.52    24.14   18.33 411
      BLR6b 39.81 25.39    24.84   18.69 411
       BLR6 39.85 25.47    24.93   18.78 411
      BLR5b 40.03 25.43    23.80   18.17 411
      BLR2b 40.05 25.19    23.83   17.98 411
      BLR3b 40.06 25.20    23.77   17.95 411
       BLR5 40.09 25.55    23.95   18.28 411
      BLR1b 40.09 26.35    26.01   19.66 411
       BLR2 40.10 25.34    23.90   18.09 411
       BLR3 40.11 25.34    23.85   18.06 411
       BLR1 40.17 26.20    25.88   19.43 411
      BLR8b 40.62 25.67    23.48   16.50 411
       BLR8 40.87 25.58    23.54   16.35 411


In [188]:
model_eval(min_PA=250)

      model  RMSE   MAE  RMSE_PA  MAE_PA   n
      BLR3b 22.02 17.22    21.50   16.66 284
       BLR3 22.07 17.28    21.52   16.72 284
      BLR2b 22.11 17.30    21.57   16.71 284
       BLR2 22.12 17.34    21.59   16.77 284
      BLR5b 22.21 17.59    21.51   16.88 284
       BLR5 22.31 17.65    21.61   16.97 284
   BMC_wRC+ 22.51 17.54    21.94   16.96 284
MARCEL_wRC+ 22.52 17.57    21.66   16.88 284
      BLR6b 22.81 17.73    22.90   17.60 284
       BLR6 22.87 17.80    22.96   17.68 284
       BLR8 23.38 17.72    21.88   15.47 284
      BLR8b 23.59 18.04    21.91   15.67 284
       BLR1 24.06 18.52    23.89   18.15 284
      BLR1b 24.46 18.91    24.12   18.43 284


In [189]:
model8

,posterior mean,lower 95% bound,upper 95% bound
x0,44.680875,16.378454,73.316511
wRC+_t1,0.527902,0.255466,0.798343
wRC+_delta,-0.128611,-0.316174,0.059826


In [190]:
model3

,posterior mean,lower 95% bound,upper 95% bound
x0,93.857555,76.202577,111.420580
BB-K%_t1,141.523559,35.826548,244.884674
Barrel%_t1,305.212983,140.251812,472.831150


In [191]:
model2

,posterior mean,lower 95% bound,upper 95% bound
x0,96.756276,66.634132,126.568334
K%_t1,-148.796712,-274.345093,-22.661785
BB%_t1,116.710044,-105.847411,334.773194
Barrel%_t1,315.400922,119.493880,507.131645


## 4. Bayesian Linear Regression (PyMC)

Six models predicting target-season wRC+ for **non-rookie, non-pitcher** players (2017–2023 training, 2024 test).  
Ohtani is retained throughout.

| Label | Predictors | Likelihood |
|---|---|---|
| `wRC_homo` | wRC+_t1 | homoskedastic |
| `wRC_hetero` | wRC+_t1 | hetero: σ/√PA |
| `wRC_delta_homo` | wRC+_t1, wRC+_delta | homoskedastic |
| `wRC_delta_hetero` | wRC+_t1, wRC+_delta | hetero: σ/√PA |
| `BBK_Brl_homo` | BB-K%_t1, Barrel%_t1 | homoskedastic |
| `BBK_Brl_hetero` | BB-K%_t1, Barrel%_t1 | hetero: σ/√PA |

Priors: intercept ~ N(100, 20), β ~ N(0, 5), σ_obs ~ HN(500).  
Model comparison via PSIS-LOO. Performance vs Bayesian Marcel and deterministic Marcel on 2024 ≥ 100 PA players.

In [192]:
import pytensor.tensor as pt

OHTANI_IDFG = 19755

# Join current-season primary_pos and rookie flag onto regression frames
pos_rook = pd.concat([
    train[["IDfg", "Season", "primary_pos", "rookie"]],
    test[["IDfg",  "Season", "primary_pos", "rookie"]],
]).drop_duplicates(subset=["IDfg", "Season"])

train_pm = train_reg.merge(pos_rook, on=["IDfg", "Season"], how="left")
test_pm  = test_reg.merge( pos_rook, on=["IDfg", "Season"], how="left")

def _pm_filter(df):
    is_ohtani  = df["IDfg"] == OHTANI_IDFG
    is_pitcher = df["primary_pos"] == "P"
    is_rookie  = df["rookie"] == 1
    return df[(~is_pitcher | is_ohtani) & (~is_rookie | is_ohtani)].copy()

train_pm = _pm_filter(train_pm)
test_pm  = _pm_filter(test_pm)

# Base training filter: 100 PA target + 100 PA prior season
_base = train_pm[(train_pm["PA"] >= 100) & (train_pm["PA_t1"] >= 100)]

# M1 / M3 specs: 2017–2023
train_m1m3 = _base[_base["Season"] <= 2023].reset_index(drop=True)

# M2 spec: requires t-2 data → 2018–2023, drop rows with missing delta
train_m2 = (
    _base[(_base["Season"] >= 2018) & (_base["Season"] <= 2023)]
    .dropna(subset=["wRC+_delta"])
    .reset_index(drop=True)
)

print(f"train_m1m3 : {len(train_m1m3):,} rows  |  seasons {sorted(train_m1m3['Season'].unique())}")
print(f"train_m2   : {len(train_m2):,} rows   |  seasons {sorted(train_m2['Season'].unique())}")
print(f"test_pm    : {len(test_pm):,} rows   |  season  {sorted(test_pm['Season'].unique())}")


train_m1m3 : 2,279 rows  |  seasons [2017, 2018, 2019, 2020, 2021, 2022, 2023]
train_m2   : 1,749 rows   |  seasons [2018, 2019, 2020, 2021, 2022, 2023]
test_pm    : 542 rows   |  season  [2024]


In [193]:
def fit_bayesian_reg(name, feature_cols, df_train, heteroskedastic=False,
                     n_draws=1000, n_tune=1000, seed=42):
    """
    Fit a Bayesian linear regression for wRC+ with PyMC.
    Priors: intercept N(100,20), beta N(0,5), sigma_obs HN(500).
    Heteroskedastic variant: obs_sd = sigma_obs / sqrt(PA).
    Returns (model, trace, meta).
    """
    work  = df_train.dropna(subset=feature_cols + ["wRC+", "PA"]).reset_index(drop=True)
    X_tr  = work[feature_cols].to_numpy(dtype=float)
    y_tr  = work["wRC+"].to_numpy(dtype=float)
    PA_tr = work["PA"].to_numpy(dtype=float)
    n_feat = X_tr.shape[1]

    print(f"\n{'─'*60}")
    print(f"  {name}  |  n={len(work):,}  |  hetero={heteroskedastic}")
    print(f"{'─'*60}")

    with pm.Model() as model:
        X_data  = pm.MutableData("X_data",  X_tr)
        y_data  = pm.MutableData("y_data",  y_tr)
        pa_data = pm.MutableData("pa_data", PA_tr)

        intercept = pm.Normal("intercept", mu=100,  sigma=20)
        beta      = pm.Normal("beta",      mu=0,    sigma=5,   shape=n_feat)
        sigma_obs = pm.HalfNormal("sigma_obs",      sigma=500)

        mu     = intercept + pm.math.dot(X_data, beta)
        obs_sd = sigma_obs / pm.math.sqrt(pa_data) if heteroskedastic else sigma_obs

        pm.Normal("y", mu=mu, sigma=obs_sd, observed=y_data)

        trace = pm.sample(
            draws=n_draws, tune=n_tune, target_accept=0.9,
            random_seed=seed, nuts_sampler="numpyro", progressbar=True,
        )
        pm.compute_log_likelihood(trace)

    summary = az.summary(trace, var_names=["intercept", "beta", "sigma_obs"], round_to=3)
    summary.index = (["intercept"]
                     + [f"beta[{c}]" for c in feature_cols]
                     + ["sigma_obs"])
    print(summary.to_string())

    meta = {
        "name": name, "feature_cols": feature_cols,
        "heteroskedastic": heteroskedastic, "n_train": len(work),
        "summary": summary,
    }
    return model, trace, meta


# Spec table: (label, features, train_df, heteroskedastic)
SPECS = [
    ("wRC_homo",         ["wRC+_t1"],                 train_m1m3, False),
    ("wRC_hetero",       ["wRC+_t1"],                 train_m1m3, True),
    ("wRC_delta_homo",   ["wRC+_t1", "wRC+_delta"],   train_m2,   False),
    ("wRC_delta_hetero", ["wRC+_t1", "wRC+_delta"],   train_m2,   True),
    ("BBK_Brl_homo",     ["BB-K%_t1", "Barrel%_t1"], train_m1m3, False),
    ("BBK_Brl_hetero",   ["BB-K%_t1", "Barrel%_t1"], train_m1m3, True),
]

fit_results = {}
for name, features, train_df, hetero in SPECS:
    mdl, trc, meta = fit_bayesian_reg(name, features, train_df, heteroskedastic=hetero)
    fit_results[name] = (mdl, trc, meta)



────────────────────────────────────────────────────────────
  wRC_homo  |  n=2,279  |  hetero=False
────────────────────────────────────────────────────────────


Compiling.. :   0%|          | 0/2000 [00:00<?, ?it/s]





Running chain 0:   0%|          | 0/2000 [00:00<?, ?it/s]


Running chain 0:  50%|█████     | 1000/2000 [00:00<00:00, 9771.17it/s]


Running chain 3: 100%|██████████| 2000/2000 [00:00<00:00, 2322.35it/s]

Running chain 2: 100%|██████████| 2000/2000 [00:00<00:00, 2190.30it/s]


                 mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  ess_tail  r_hat
intercept      53.473  2.147  49.651   57.545      0.056    0.040  1471.167  1869.635  1.001
beta[wRC+_t1]   0.438  0.020   0.400    0.474      0.001    0.000  1488.213  1792.070  1.001
sigma_obs      26.120  0.382  25.348   26.789      0.009    0.006  1786.580  1826.879  1.002

────────────────────────────────────────────────────────────
  wRC_hetero  |  n=2,279  |  hetero=True
────────────────────────────────────────────────────────────


Compiling.. :   0%|          | 0/2000 [00:00<?, ?it/s]





Running chain 0:   0%|          | 0/2000 [00:00<?, ?it/s]




Running chain 0:  60%|██████    | 1200/2000 [00:00<00:00, 5678.02it/s]


Running chain 0:  95%|█████████▌| 1900/2000 [00:00<00:00, 6007.84it/s]

Running chain 3: 100%|██████████| 2000/2000 [00:00<00:00, 2024.44it/s]


Running chain 1: 100%|██████████| 2000/2000 [00:01<00:00, 1708.39it/s]


                  mean     sd   hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  ess_tail  r_hat
intercept       57.213  2.031   53.466   60.805      0.058    0.041  1232.524  1339.659  1.002
beta[wRC+_t1]    0.442  0.019    0.409    0.476      0.001    0.000  1224.531  1261.804  1.001
sigma_obs      470.179  7.105  456.687  483.548      0.165    0.116  1884.634  1743.527  1.003

────────────────────────────────────────────────────────────
  wRC_delta_homo  |  n=1,749  |  hetero=False
────────────────────────────────────────────────────────────


...

                     mean     sd   hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  ess_tail  r_hat
intercept          49.292  2.498   44.886   54.363      0.065    0.046  1483.310  1816.438  1.001
beta[wRC+_t1]       0.521  0.023    0.477    0.564      0.001    0.000  1486.893  1872.371  1.001
beta[wRC+_delta]   -0.120  0.017   -0.151   -0.090      0.000    0.000  2035.935  2259.005  1.001
sigma_obs         465.124  7.948  449.657  479.578      0.165    0.116  2334.903  1895.346  1.001

────────────────────────────────────────────────────────────
  BBK_Brl_homo  |  n=2,279  |  hetero=False
────────────────────────────────────────────────────────────


Compiling.. :   0%|          | 0/2000 [00:00<?, ?it/s]





Running chain 0:   0%|          | 0/2000 [00:00<?, ?it/s]




Running chain 1: 100%|██████████| 2000/2000 [00:01<00:00, 1945.43it/s]


Running chain 0: 100%|██████████| 2000/2000 [00:01<00:00, 1894.75it/s]

Running chain 2: 100%|██████████| 2000/2000 [00:01<00:00, 1812.66it/s]


                    mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  ess_tail  r_hat
intercept         98.843  0.907  97.188  100.582      0.019    0.013  2378.610  2346.157  1.002
beta[BB-K%_t1]    19.400  4.429  10.721   27.223      0.088    0.063  2551.485  2610.006  1.001
beta[Barrel%_t1]  27.088  4.749  18.279   35.863      0.085    0.060  3143.963  2831.894  1.001
sigma_obs         28.121  0.424  27.324   28.918      0.007    0.005  3718.019  2707.755  1.001

────────────────────────────────────────────────────────────
  BBK_Brl_hetero  |  n=2,279  |  hetero=True
────────────────────────────────────────────────────────────


Compiling.. :   0%|          | 0/2000 [00:00<?, ?it/s]





Running chain 0:   0%|          | 0/2000 [00:00<?, ?it/s]





Running chain 0:   5%|▌         | 100/2000 [00:01<00:08, 211.99it/s]


Running chain 1: 100%|██████████| 2000/2000 [00:01<00:00, 1455.43it/s]


                     mean     sd   hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  ess_tail  r_hat
intercept         104.251  0.852  102.663  105.906      0.017    0.012  2425.028  2567.497  1.001
beta[BB-K%_t1]     21.352  4.365   12.800   29.106      0.084    0.059  2705.769  2896.665  1.001
beta[Barrel%_t1]   31.790  4.706   23.172   40.578      0.086    0.061  2993.660  2786.639  1.001
sigma_obs         511.154  7.629  496.785  525.388      0.132    0.093  3349.595  2830.459  1.000


In [194]:
loo_rows = []
for name, (mdl, trc, meta) in fit_results.items():
    r = az.loo(trc)
    loo_rows.append({
        "model":      name,
        "predictors": " + ".join(meta["feature_cols"]),
        "likelihood": "hetero" if meta["heteroskedastic"] else "homo",
        "n_train":    meta["n_train"],
        "elpd_loo":   round(float(r.elpd_loo), 1),
        "se":         round(float(r.se),        1),
        "p_loo":      round(float(r.p_loo),     1),
    })

loo_df = (
    pd.DataFrame(loo_rows)
    .sort_values("elpd_loo", ascending=False)
    .reset_index(drop=True)
)
print("PSIS-LOO expected log predictive density (higher = better fit)")
print("Note: wRC_delta models trained on 2018–2023; others on 2017–2023.")
print("      LOO values are not directly comparable across different training sets.\n")
print(loo_df.to_string(index=False))


PSIS-LOO expected log predictive density (higher = better fit)
Note: wRC_delta models trained on 2018–2023; others on 2017–2023.
      LOO values are not directly comparable across different training sets.

           model            predictors likelihood  n_train  elpd_loo   se  p_loo
wRC_delta_hetero  wRC+_t1 + wRC+_delta     hetero     1749   -8114.3 33.3    5.0
  wRC_delta_homo  wRC+_t1 + wRC+_delta       homo     1749   -8165.1 34.7    5.0
      wRC_hetero               wRC+_t1     hetero     2279  -10591.9 37.4    3.4
        wRC_homo               wRC+_t1       homo     2279  -10670.2 38.4    3.4
  BBK_Brl_hetero BB-K%_t1 + Barrel%_t1     hetero     2279  -10782.9 38.7    2.8
    BBK_Brl_homo BB-K%_t1 + Barrel%_t1       homo     2279  -10838.3 37.6    2.7


In [195]:
def predict_pm(trace, meta, test_df, hdi_prob=0.94, seed=0):
    """
    Point predictions (posterior mean mu) and equal-tail HDI bounds
    for the posterior predictive distribution on test_df.
    Returns three Series indexed by IDfg: (point_pred, hdi_lo, hdi_hi).
    """
    feature_cols    = meta["feature_cols"]
    heteroskedastic = meta["heteroskedastic"]

    work = test_df.dropna(subset=feature_cols + ["PA"]).copy()
    if len(work) == 0:
        empty = pd.Series(dtype=float)
        return empty, empty, empty

    X_te  = work[feature_cols].to_numpy(dtype=float)
    PA_te = work["PA"].to_numpy(dtype=float)

    intercept_d = trace.posterior["intercept"].values.flatten()
    beta_d      = trace.posterior["beta"].values.reshape(-1, len(feature_cols))
    sigma_d     = trace.posterior["sigma_obs"].values.flatten()
    S = len(intercept_d)

    mu_samps = intercept_d[:, None] + beta_d @ X_te.T   # (S, n_te)

    if heteroskedastic:
        sigma_eff = sigma_d[:, None] / np.sqrt(PA_te[None, :])
    else:
        sigma_eff = np.tile(sigma_d[:, None], (1, len(work)))

    rng    = np.random.default_rng(seed)
    y_pred = mu_samps + rng.standard_normal(mu_samps.shape) * sigma_eff  # (S, n_te)

    alpha  = (1 - hdi_prob) / 2
    lo_pct = alpha * 100
    hi_pct = (1 - alpha) * 100

    idx = work["IDfg"].values
    return (
        pd.Series(mu_samps.mean(axis=0),                        index=idx),
        pd.Series(np.percentile(y_pred, lo_pct, axis=0).round(1), index=idx),
        pd.Series(np.percentile(y_pred, hi_pct, axis=0).round(1), index=idx),
    )


pred_store = {}
hdi_store  = {}

for name, (mdl, trc, meta) in fit_results.items():
    pt_pred, lo, hi = predict_pm(trc, meta, test_pm)
    pred_store[name] = pt_pred.round(1)
    hdi_store[name]  = (lo, hi)
    print(f"{name}: {pt_pred.notna().sum()} predictions")


wRC_homo: 515 predictions
wRC_hetero: 515 predictions
wRC_delta_homo: 418 predictions
wRC_delta_hetero: 418 predictions


/var/folders/3f/5z6bhnqn1fv15360tr2wfskr0000gn/T/ipykernel_39206/1954214022.py:26: RuntimeWarning: divide by zero encountered in divide
  sigma_eff = sigma_d[:, None] / np.sqrt(PA_te[None, :])
/Users/johannhatzius/Library/Python/3.9/lib/python/site-packages/numpy/lib/function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)
/var/folders/3f/5z6bhnqn1fv15360tr2wfskr0000gn/T/ipykernel_39206/1954214022.py:26: RuntimeWarning: divide by zero encountered in divide
  sigma_eff = sigma_d[:, None] / np.sqrt(PA_te[None, :])
/Users/johannhatzius/Library/Python/3.9/lib/python/site-packages/numpy/lib/function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


BBK_Brl_homo: 514 predictions
BBK_Brl_hetero: 514 predictions


/var/folders/3f/5z6bhnqn1fv15360tr2wfskr0000gn/T/ipykernel_39206/1954214022.py:26: RuntimeWarning: divide by zero encountered in divide
  sigma_eff = sigma_d[:, None] / np.sqrt(PA_te[None, :])
/Users/johannhatzius/Library/Python/3.9/lib/python/site-packages/numpy/lib/function_base.py:4655: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = subtract(b, a)


In [196]:
# Extend eval_df with PyMC point predictions
eval_ext = eval_df[["IDfg", "Name", "Age", "primary_pos", "PA", "wRC+",
                     "MARCEL_wRC+", "BMC_wRC+"]].copy()
for name, pred in pred_store.items():
    eval_ext[name] = eval_ext["IDfg"].map(pred)

# Restrict to >= 100 PA
ev100  = eval_ext[eval_ext["PA"] >= 100].copy()
actual = ev100["wRC+"]

model_cols = ["MARCEL_wRC+", "BMC_wRC+"] + list(fit_results.keys())
rows = []
for col in model_cols:
    valid = ev100[col].notna()
    resid = actual[valid] - ev100.loc[valid, col]
    rows.append({
        "model": col,
        "n":     int(valid.sum()),
        "RMSE":  round(float(np.sqrt((resid**2).mean())), 2),
        "MAE":   round(float(resid.abs().mean()),          2),
    })

rmse_df = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
print("RMSE / MAE — 2024 players with ≥ 100 PA (lower is better)\n")
print(rmse_df.to_string(index=False))


RMSE / MAE — 2024 players with ≥ 100 PA (lower is better)

           model   n  RMSE   MAE
  wRC_delta_homo 314 25.38 19.20
        BMC_wRC+ 358 25.60 19.76
wRC_delta_hetero 314 25.62 19.46
     MARCEL_wRC+ 358 25.84 19.83
        wRC_homo 358 26.64 20.51
      wRC_hetero 358 27.00 20.80
    BBK_Brl_homo 358 28.16 21.45
  BBK_Brl_hetero 358 28.80 22.08


In [197]:
# 94% posterior predictive HDI coverage — 2024 ≥ 100 PA players
cov_rows = []

# Bayesian Marcel (HDI bounds already in bmc_projections)
bmc_ev = (
    ev100[["IDfg", "wRC+"]]
    .merge(bmc_projections[["IDfg", "pred_hdi_lo", "pred_hdi_hi"]], on="IDfg", how="left")
    .dropna(subset=["wRC+", "pred_hdi_lo"])
)
inside_bmc = (bmc_ev["wRC+"] >= bmc_ev["pred_hdi_lo"]) &              (bmc_ev["wRC+"] <= bmc_ev["pred_hdi_hi"])
cov_rows.append({"model": "BMC_wRC+", "n": len(bmc_ev),
                 "coverage_94": round(float(inside_bmc.mean()), 3)})

# 6 PyMC models
for name, (lo_s, hi_s) in hdi_store.items():
    ev = ev100[["IDfg", "wRC+"]].copy()
    ev["hdi_lo"] = ev["IDfg"].map(lo_s)
    ev["hdi_hi"] = ev["IDfg"].map(hi_s)
    ev = ev.dropna(subset=["wRC+", "hdi_lo"])
    inside = (ev["wRC+"] >= ev["hdi_lo"]) & (ev["wRC+"] <= ev["hdi_hi"])
    cov_rows.append({"model": name, "n": len(ev),
                     "coverage_94": round(float(inside.mean()), 3)})

cov_df = (
    pd.DataFrame(cov_rows)
    .sort_values("coverage_94", ascending=False)
    .reset_index(drop=True)
)
print("94% posterior predictive HDI coverage — 2024 ≥ 100 PA players")
print("(well-calibrated models should be close to 0.94)\n")
print(cov_df.to_string(index=False))


94% posterior predictive HDI coverage — 2024 ≥ 100 PA players
(well-calibrated models should be close to 0.94)

           model   n  coverage_94
  BBK_Brl_hetero 358        0.947
  wRC_delta_homo 314        0.939
wRC_delta_hetero 314        0.939
      wRC_hetero 358        0.933
        wRC_homo 358        0.927
    BBK_Brl_homo 358        0.922
        BMC_wRC+ 358        0.883


In [ ]:
# ── 2024 output table ────────────────────────────────────────────────────────
# Two BMC variants:
#   BMC          — SD is the full posterior predictive (theta uncertainty + obs noise)
#   BMC_theta    — SD is theta_aged SD only (parameter uncertainty, no obs noise)
# All PyMC model SDs are full posterior predictive SDs, consistent with 94% HDI.

TARGET_MODELS = ["wRC_delta_homo", "wRC_homo", "BBK_Brl_homo"]

out = (
    test[["IDfg", "Name", "team_start", "primary_pos", "role", "Age", "rookie", "wRC+"]]
    .drop_duplicates("IDfg")
    .copy()
)

rng = np.random.default_rng(0)

# ── PyMC models ───────────────────────────────────────────────────────────────
for name in TARGET_MODELS:
    _, trc, meta = fit_results[name]
    feat   = meta["feature_cols"]
    hetero = meta["heteroskedastic"]

    work  = test_pm.dropna(subset=feat + ["PA"]).copy()
    X_te  = work[feat].to_numpy(dtype=float)
    PA_te = work["PA"].to_numpy(dtype=float)

    intercept_d = trc.posterior["intercept"].values.flatten()
    beta_d      = trc.posterior["beta"].values.reshape(-1, len(feat))
    sigma_d     = trc.posterior["sigma_obs"].values.flatten()

    mu_samps  = intercept_d[:, None] + beta_d @ X_te.T
    sigma_eff = (sigma_d[:, None] / np.sqrt(PA_te[None, :])
                 if hetero else sigma_d[:, None])
    y_pred    = mu_samps + rng.standard_normal(mu_samps.shape) * sigma_eff

    idx = work["IDfg"].values
    out[f"{name}_mean"] = out["IDfg"].map(pd.Series(y_pred.mean(axis=0).round(1), index=idx))
    out[f"{name}_sd"]   = out["IDfg"].map(pd.Series(y_pred.std(axis=0).round(1),  index=idx))

# ── Bayesian Marcel — full posterior predictive SD (BMC) ──────────────────────
sigma_obs_bmc = trace.posterior["sigma_obs"].values.flatten()           # (S,)
pa_2023  = train[train["Season"] == 2023].set_index("IDfg")["PA"]
proj_pa  = proj_df["IDfg"].map(pa_2023).fillna(400).values

sigma_eff_bmc = sigma_obs_bmc[:, None] / np.sqrt(proj_pa[None, :])     # (S, n_proj)
y_pred_bmc    = (theta_proj_aged
                 + rng.standard_normal(theta_proj_aged.shape) * sigma_eff_bmc)

bmc_idx    = proj_df["IDfg"].values
bmc_mean_s = pd.Series(y_pred_bmc.mean(axis=0).round(1), index=bmc_idx)
bmc_sd_s   = pd.Series(y_pred_bmc.std(axis=0).round(1),  index=bmc_idx)

out["BMC_mean"]    = out["IDfg"].map(bmc_mean_s)
out["BMC_pred_sd"] = out["IDfg"].map(bmc_sd_s)

# ── Bayesian Marcel — theta_aged SD only (BMC_theta) ─────────────────────────
# SD = std(theta_proj_aged) across posterior draws — parameter uncertainty only,
# without adding observation noise. Mean is identical to BMC.
bmc_theta_sd_s = pd.Series(theta_proj_aged.std(axis=0).round(1), index=bmc_idx)
out["BMC_theta_mean"] = out["IDfg"].map(bmc_mean_s)   # identical mean
out["BMC_theta_sd"]   = out["IDfg"].map(bmc_theta_sd_s)

# ── Deterministic MARCEL — point estimate only ────────────────────────────────
out["MARCEL_wRC+"] = out["IDfg"].map(marcel_det_preds.set_index("IDfg")["MARCEL_wRC+"])

out = out.rename(columns={"wRC+": "actual_wRC+"})
out = out[[
    "Name", "team_start", "Age", "rookie", "primary_pos", "role", "actual_wRC+",
    "wRC_delta_homo_mean", "wRC_delta_homo_sd",
    "BMC_mean", "BMC_pred_sd",
    "BMC_theta_mean", "BMC_theta_sd",
    "wRC_homo_mean", "wRC_homo_sd",
    "BBK_Brl_homo_mean", "BBK_Brl_homo_sd",
    "MARCEL_wRC+",
]]

out_path = data_path / "data" / "model_outputs" / "regression_outputs_2024.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(out_path, index=False)

print(f"Saved {len(out):,} rows  →  {out_path}")
print(f"Columns: {out.columns.tolist()}")
out.head(10)